## Импорты, создание датасетов, проверка датасетов, настройка конфигурации

#### Импорты

In [28]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from ETL import ETL_function, date_split
from create_candidates import create_candidates

### Конфигурации

In [29]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", 100)

In [30]:
path_check = r"data/after_transform_xlsx/dataset.xlsx"
path_data = r"data/after_transform_csv/dataset.csv"

path_data1 = r"data/after_transform_csv/dataset1.csv"
path_data2 = r"data/after_transform_csv/dataset2.csv"
path_data3 = r"data/after_transform_csv/dataset3.csv"
path_data4 = r"data/after_transform_csv/dataset4.csv"
path_data5 = r"data/after_transform_csv/dataset5.csv"

path_candidates1 = r"data/candidates/dataset1_candidates.csv"
path_candidates2 = r"data/candidates/dataset2_candidates.csv"
path_candidates3 = r"data/candidates/dataset3_candidates.csv"
path_candidates4 = r"data/candidates/dataset4_candidates.csv"
path_candidates5 = r"data/candidates/dataset5_candidates.csv"
path_submit_candidates = r"data/candidates/submit_candidates.csv"

### Составим excel файл, для проверки правильности заполнения данных

In [31]:
# ETL_function(
#     cyclecally_by_month=False,
#     path_save=path_check,
#     num_months=1,
#     add_negatives=True,
#     neg_min=5,
#     neg_max=10,
#     neg_train_share=0.5,
#     random_state=42,
#     rank_lambda=0.5,
#     sim_noise_scale=0.01,
# )

In [32]:
# df = pd.read_excel(r"data/after_transform_xlsx/dataset.xlsx")

### Создание датасетов, для обучения

In [33]:
# train
paths_train = [path_data5]
flag_train = 0 if all(os.path.exists(p) for p in paths_train) else 1

# validation + submit (только для 5-го месяца)
paths_can = [path_candidates5, path_submit_candidates]
flag_can = 0 if all(os.path.exists(p) for p in paths_can) else 1

months_to_build = [5]
build_submit = True

if flag_train:
    ETL_function(cyclecally_by_month=False, num_months=5)

if flag_can:
    create_candidates(
        total_candidates=200,
        pos_limit=100,
        neg_train_share=0.5,
        random_state=42,
        rank_lambda=0.5,
        sim_noise_scale=0.01,
        months=months_to_build,
        build_submit=build_submit,
    )

Запуск Extract
Датафрейм сформировался 

Запуск Transform
Трансформации выполнены 

Запуск Load
Датафрейм сохранен по пути: data/after_transform_csv/dataset.csv


In [34]:
df_train = pd.read_csv(path_data)

In [35]:
df_val = pd.read_csv(path_candidates5)

### Функция для расчета score

In [36]:
def build_edition_genres(editions, book_genres):
    return (
        editions[["edition_id", "book_id"]]
        .merge(book_genres, on="book_id", how="left")
        .groupby("edition_id")["genre_id"]
        .apply(lambda x: set(x.dropna().tolist()))
        .to_dict()
    )


def rerank_with_diversity(df_pred, edition_genres, top_n=30, k=20, alpha=0.7, beta=0.5):
    rows = []
    for user_id, g in df_pred.groupby("user_id"):
        g = g.sort_values("score", ascending=False).head(top_n).copy()

        scores = g["score"].to_numpy()
        s_min, s_max = scores.min(), scores.max()
        if s_max > s_min:
            g["rel_proxy"] = (g["score"] - s_min) / (s_max - s_min)
        else:
            g["rel_proxy"] = 0.5

        selected = []
        seen_genres = set()

        while len(selected) < min(k, len(g)):
            best_idx = None
            best_gain = -1.0
            for idx, row in g.iterrows():
                if idx in selected:
                    continue

                rel = float(row["rel_proxy"])
                genres = edition_genres.get(int(row["edition_id"]), set())

                if genres:
                    cov_gain = len(genres - seen_genres) / len(genres)
                else:
                    cov_gain = 0.0

                if not selected:
                    ild_gain = 0.0
                else:
                    dists = []
                    for s_idx in selected:
                        s_eid = int(g.loc[s_idx, "edition_id"])
                        g1 = genres
                        g2 = edition_genres.get(s_eid, set())
                        u = g1 | g2
                        d = 0.0 if len(u) == 0 else 1.0 - (len(g1 & g2) / len(u))
                        dists.append(d)
                    ild_gain = float(np.mean(dists)) if dists else 0.0

                div_gain = beta * cov_gain + (1 - beta) * ild_gain
                total_gain = alpha * rel + (1 - alpha) * div_gain

                if total_gain > best_gain:
                    best_gain = total_gain
                    best_idx = idx

            selected.append(best_idx)
            sel_eid = int(g.loc[best_idx, "edition_id"])
            seen_genres |= edition_genres.get(sel_eid, set())

        for rank, idx in enumerate(selected, start=1):
            rows.append({
                "user_id": user_id,
                "edition_id": int(g.loc[idx, "edition_id"]),
                "rank": rank,
            })

    return pd.DataFrame(rows)


def compute_score(preds, interactions, editions, book_genres, top_k=20, alpha=0.7, beta=0.5):
    rel_map = {1: 1, 2: 2}

    rel_df = (
        interactions[interactions["event_type"].isin([1, 2])]
        .groupby(["user_id", "edition_id"])["event_type"]
        .max()
        .map(rel_map)
        .reset_index(name="rel")
    )

    rel_by_user = rel_df.groupby("user_id").apply(
        lambda x: dict(zip(x["edition_id"], x["rel"]))
    ).to_dict()

    ed_genres = build_edition_genres(editions, book_genres)

    preds_sorted = preds.sort_values(["user_id", "rank"])
    pred_lists = preds_sorted.groupby("user_id")["edition_id"].apply(list).to_dict()

    w = 1.0 / np.log2(np.arange(1, top_k + 1) + 1)
    w_sum = w.sum()

    ndcgs = []
    diversities = []

    for user_id, recs in pred_lists.items():
        recs = recs[:top_k]
        rel_user = rel_by_user.get(user_id, {})

        rels = np.array([rel_user.get(eid, 0) for eid in recs], dtype=float)
        dcg = (rels * w[:len(rels)]).sum()

        ideal_rels = sorted(rel_user.values(), reverse=True)[:top_k]
        if len(ideal_rels) < top_k:
            ideal_rels = ideal_rels + [0] * (top_k - len(ideal_rels))
        ideal_rels = np.array(ideal_rels, dtype=float)
        idcg = (ideal_rels * w).sum()

        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcgs.append(ndcg)

        seen = set()
        cov = 0.0
        rel_indices = []
        rel_genres = []

        for k, eid in enumerate(recs, start=1):
            rel = rel_user.get(eid, 0)
            if rel > 0:
                genres = ed_genres.get(eid, set())
                if genres:
                    new_genres = genres - seen
                    cov += (1.0 / np.log2(k + 1)) * (len(new_genres) / len(genres))
                    seen |= genres
                rel_indices.append(eid)
                rel_genres.append(genres)

        coverage = cov / w_sum if w_sum > 0 else 0.0

        m = len(rel_indices)
        if m < 2:
            ild = 0.0
        else:
            s = 0.0
            for i in range(m):
                for j in range(i + 1, m):
                    a = rel_genres[i]
                    b = rel_genres[j]
                    u = a | b
                    d = 0.0 if len(u) == 0 else 1.0 - (len(a & b) / len(u))
                    s += d
            ild = 2.0 * s / (m * (m - 1))

        diversity = beta * coverage + (1 - beta) * ild
        diversities.append(diversity)

    mean_ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0
    mean_div = float(np.mean(diversities)) if diversities else 0.0
    return alpha * mean_ndcg + (1 - alpha) * mean_div


## Обучение модели

In [37]:
group_col = "user_id"
target_col = "label"

cat_features = ["main_genre", "language_id", "author_cluster", "user_cluster", "publication_year"]

# сортировка по группе обязательна
df_train = df_train.sort_values(group_col)
df_val   = df_val.sort_values(group_col)

# признаки
features = [c for c in df_train.columns if c not in [target_col, group_col, "edition_id"]]

# убедимся, что категориальные в category dtype
for c in cat_features:
    if c in df_train.columns:
        df_train[c] = df_train[c].astype("category")
        df_val[c]   = df_val[c].astype("category")

# group sizes (сколько строк на user)
train_group = df_train.groupby(group_col).size().to_numpy()
val_group   = df_val.groupby(group_col).size().to_numpy()

X_train = df_train[features]
y_train = df_train[target_col].astype(float)

X_val = df_val[features]
y_val = df_val[target_col].astype(float)

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    group=train_group,
    categorical_feature=[c for c in cat_features if c in features],
    free_raw_data=False,
)

val_set = lgb.Dataset(
    X_val,
    label=y_val,
    group=val_group,
    categorical_feature=[c for c in cat_features if c in features],
    free_raw_data=False,
)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [30],
    "learning_rate": 0.01,
    "num_leaves": 63,          # аналог depth/complexity
    "min_data_in_leaf": 100,
    "feature_fraction": 0.6,
    "bagging_fraction": 0.6,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
    "num_threads": -1,
}

model = lgb.train(
    params,
    train_set,
    num_boost_round=5000,
    valid_sets=[val_set],
    valid_names=["val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=50),
    ],
)

df_val["pred"] = model.predict(X_val, num_iteration=model.best_iteration)

Training until validation scores don't improve for 100 rounds
[50]	val's ndcg@30: 0.468599
[100]	val's ndcg@30: 0.473855
Early stopping, best iteration is:
[1]	val's ndcg@30: 0.811809


In [38]:
# -----------------------------
# rerank + score
# -----------------------------
editions = pd.read_csv(r"data/data/editions.csv")
book_genres = pd.read_csv(r"data/data/book_genres.csv")
interactions = pd.read_csv(r"data/data/interactions.csv")
interactions["event_ts"] = pd.to_datetime(interactions["event_ts"])

edition_genres = build_edition_genres(editions, book_genres)

df_pred = df_val[[group_col, "edition_id"]].copy()
df_pred["score"] = df_val["pred"]

df_rerank = rerank_with_diversity(df_pred, edition_genres, top_n=30, k=20)

val_month_index = 5
val_start = date_split[val_month_index - 1]
val_end = date_split[val_month_index] if val_month_index < len(date_split) else None
if val_end is None:
    val_end = interactions["event_ts"].max()
val_interactions = interactions[(interactions["event_ts"] >= val_start) & (interactions["event_ts"] < val_end)]

score = compute_score(df_rerank, val_interactions, editions, book_genres)
score

0.5096796479487621

In [39]:
df_train.groupby("user_id").size().reset_index(name="cnt").sort_values(by=["cnt"])

,user_id,cnt
763,783950,26
2841,6746080,26
3608,8518270,26
1903,3150610,28
1528,1772230,28
...,...,...
3462,8196010,8250
4823,11578480,8332
2537,5718550,8425
4501,11053490,8459


## Формирование Submition

In [41]:
# -----------------------------
# submission
# -----------------------------

df_submit = pd.read_csv(path_submit_candidates)

# те же фичи, что в трейне (features, cat_features уже определены)
for c in cat_features:
    if c in df_submit.columns:
        df_submit[c] = df_submit[c].astype("category")

X_submit = df_submit[features]

df_submit["pred"] = model.predict(X_submit, num_iteration=model.best_iteration)

df_submit_pred = df_submit[[group_col, "edition_id"]].copy()
df_submit_pred["score"] = df_submit["pred"]

edition_genres = build_edition_genres(editions, book_genres)
df_submission = rerank_with_diversity(df_submit_pred, edition_genres, top_n=30, k=20)

df_submission.to_csv("submission.csv", index=False)
df_submission.head()

,user_id,edition_id,rank
0,560,1001514720,1
1,560,1006232332,2
2,560,1003618326,3
3,560,1006232329,4
4,560,1008363562,5
